# E1 — Esercitazione: pulire un dataset nuovo (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/E1_pulizia/E1_soluzioni.ipynb)

## Come funziona un'esercitazione

Rispetto alle lezioni, qui hai **meno indicazioni**: per ogni passo trovi l'obiettivo e un **controllo** per verificare di esserci arrivato. Cosa cercare, sta a te (Google, Gemini, le dispense, i notebook delle lezioni).

- 👥 Si lavora in **gruppi da 3**, con tre ruoli che **ruotano a ogni passo**: chi **scrive** (condivide lo schermo e scrive il codice), chi **cerca** (Google, Gemini, dispense), chi **controlla** (il risultato ha senso? il numero torna?).
- Se il controllo non torna, **non andare avanti**: un errore all'inizio si porta dietro tutto il resto.
- **File → Salva una copia in Drive** per non perdere il lavoro.

## Contesto

Un'azienda alimentare ti manda le sue esportazioni mensili del 2022 e del 2023, per paese e prodotto: `esportazioni.xlsx`. Vogliono sapere quanto hanno esportato in ogni paese. Prima, però, i dati vanno puliti: è il lavoro della L3, su un file che non hai mai visto.

Leggi la descrizione delle colonne in [`dati/README.md`](https://github.com/andrealorenzon/SIMA2026/blob/main/dati/README.md): ti servirà.

⏱️ **Prima di iniziare:** se non hai finito il task della L3, completalo ora (circa 20 minuti). Questa esercitazione usa le stesse tecniche su un file nuovo.

## Preparazione

Esegui questa cella così com'è: carica le librerie e il file delle esportazioni nella tabella `ex`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ex = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/esportazioni.xlsx")

## Parte 1 — Pulizia

### 1. Conosci il file

Guarda le prime righe, le dimensioni e i tipi delle colonne. Scrivi qui sotto **tutti** i problemi che vedi, prima di correggere qualunque cosa.

In [ ]:
print(ex.shape)
print(ex.dtypes)
print(ex['paese'].value_counts())
ex.head(10)

*Risposta:* Righe duplicate; `paese` scritto in molti modi (italiano, inglese, sigle, maiuscole, spazi); `valore` è testo con il simbolo €, il punto delle migliaia, la virgola decimale e dei trattini; `mese` è testo (`"set 2022"`); alcune quantità sono negative.

### 2. Duplicati

Togli le righe duplicate.

✅ *Controllo:* 413 righe.

In [ ]:
ex = ex.drop_duplicates()
len(ex)

### 3. Paesi

Fai in modo che ogni paese sia scritto in un solo modo, in italiano.

✅ *Controllo:* `ex["paese"].nunique()` dà 5.

In [ ]:
ex["paese"] = ex["paese"].str.strip().replace({
    "Germany": "Germania", "germania": "Germania", "DE": "Germania",
    "FRANCIA": "Francia", "France": "Francia",
    "USA": "Stati Uniti", "U.S.A.": "Stati Uniti",
})
print(ex["paese"].nunique())
ex["paese"].value_counts()

### 4. Valore

Trasforma `valore` in numero (euro). Prima conta quante celle contengono solo un trattino `"-"`: dopo la conversione devono essere gli unici valori mancanti.

✅ *Controllo:* tanti valori mancanti quanti erano i trattini (9), e il primo valore della tabella è un numero con i decimali giusti.

In [ ]:
print((ex["valore"] == "-").sum())
ex["valore"] = pd.to_numeric(
    ex["valore"].str.replace("€", "", regex=False).str.strip()
    .str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)
print(ex["valore"].isna().sum())
ex["valore"].head()

### 5. Mese

Trasforma `mese` (per esempio `"set 2022"`) in una vera data, il primo giorno del mese. Python non conosce le abbreviazioni dei mesi in italiano: il dizionario qui sotto può aiutarti.

✅ *Controllo:* 24 mesi diversi, nessuna data mancante, dal gennaio 2022 al dicembre 2023.

🔎 *Da cercare:* `pandas str split expand` · `pandas map dizionario` · `pandas to_datetime`

In [ ]:
mesi = {"gen": 1, "feb": 2, "mar": 3, "apr": 4, "mag": 5, "giu": 6,
        "lug": 7, "ago": 8, "set": 9, "ott": 10, "nov": 11, "dic": 12}
parti = ex["mese"].str.split(" ", expand=True)
ex["data"] = pd.to_datetime(dict(year=parti[1].astype(int), month=parti[0].map(mesi), day=1))
print(ex["data"].nunique(), ex["data"].isna().sum(), ex["data"].min(), ex["data"].max())

### 6. Quantità negative

Quante righe hanno una quantità negativa? Sono errori da correggere? Leggi la descrizione delle colonne prima di decidere, e scrivi la tua decisione.

In [ ]:
(ex["quantita"] < 0).sum()

*Risposta:* 11 righe. Secondo la descrizione dei dati sono **resi**: merce tornata indietro. Non sono errori e vanno tenute, altrimenti il totale esportato risulterebbe gonfiato. Leggere la documentazione evita di "correggere" dati giusti.

## Parte 2 — Le domande dell'azienda

### 7. Per paese

Quanto ha esportato l'azienda in ogni paese nel 2023, in euro? Ordina dal paese più importante.

✅ *Controllo:* 5 righe.

In [ ]:
ex[ex["data"].dt.year == 2023].groupby("paese")["valore"].sum().sort_values(ascending=False).round(0)

### 8. Nel tempo

Disegna l'andamento del valore esportato totale, mese per mese. C'è un periodo dell'anno in cui si esporta di più?

In [ ]:
ex.groupby("data")["valore"].sum().plot(marker="o")
plt.title("Esportazioni mensili")
plt.ylabel("Euro")
plt.show()

*Risposta:* Dicembre spicca in entrambi gli anni (le feste); negli altri mesi le oscillazioni sono più contenute.

### 9. Un controllo di plausibilità

Calcola il prezzo medio implicito di ogni prodotto (valore totale diviso quantità totale). I prezzi hanno senso per quei prodotti?

In [ ]:
p = ex.groupby("prodotto")[["valore", "quantita"]].sum()
(p["valore"] / p["quantita"]).round(2)

*Risposta:* Sì: circa 1,8 € per la pasta, 2,4 € per la passata, 9,5 € per l'olio, 21 € per il parmigiano. Se avessimo sbagliato la conversione del valore (per esempio scambiando punto e virgola), qui verrebbero prezzi assurdi.

## Le tue decisioni

Scrivi la tabella delle decisioni di pulizia che hai preso (problema, decisione, verifica), come nel debrief della L3.

*Risposta:* | Problema | Decisione | Verifica |
|---|---|---|
| 12 righe duplicate | eliminate | 413 righe |
| paesi scritti in 13 modi | uniformati in italiano | 5 paesi |
| valore come testo con €, punti e virgole | convertito; "-" → mancante | 9 mancanti |
| mese come testo italiano | convertito in data con un dizionario | 24 mesi |
| 11 quantità negative | tenute: sono resi (dalla descrizione dei dati) | — |